In [1]:
import pyspark as py

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("olist-gold-account") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

df_orders = spark.read.parquet("../data/silver/orders/")
df_customers = spark.read.parquet("../data/silver/customers/")
df_items = spark.read.parquet("../data/silver/order_items/")
df_payments = spark.read.parquet("../data/silver/payments/")
df_reviews = spark.read.parquet("../data/silver/reviews/")
df_products = spark.read.parquet("../data/silver/products/")
df_sellers = spark.read.parquet("../data/silver/sellers/")
df_geo = spark.read.parquet("../data/silver/geolocation/")
df_pcnt = spark.read.parquet("../data/silver/category_translation/")

dataframes = {
    "orders": df_orders,
    "customers": df_customers,
    "items": df_items,
    "payments": df_payments,
    "reviews": df_reviews,
    "products": df_products,
    "sellers": df_sellers,
    "geo": df_geo,
    "pcnt": df_pcnt
}

In [6]:
print(df_orders.count())
df_orders.groupBy("order_status").count().show()

99441
+------------+-----+
|order_status|count|
+------------+-----+
|     shipped| 1107|
|    canceled|  625|
|    invoiced|  314|
|     created|    5|
|   delivered|96478|
| unavailable|  609|
|  processing|  301|
|    approved|    2|
+------------+-----+



In [10]:
from pyspark.sql.functions import col, sum as _sum, round as _round

# Trois segments selon le statut de la commande
df_orders_delivered = df_orders.filter(col("order_status") == "delivered")

df_orders_in_transit = df_orders.filter(
    col("order_status").isin(["created", "approved", "processing", "invoiced", "shipped"])
)

df_orders_excluded = df_orders.filter(
    col("order_status").isin(["canceled", "unavailable"])
)

# Jointure de chaque segment avec order_items, en conservant la granularité par article
df_gold_revenue_delivered = df_items.join(
    df_orders_delivered.select("order_id", "customer_id", "order_purchase_timestamp"),
    "order_id", "inner"
)

df_gold_revenue_in_transit = df_items.join(
    df_orders_in_transit.select("order_id", "customer_id", "order_purchase_timestamp"),
    "order_id", "inner"
)

df_gold_revenue_excluded = df_items.join(
    df_orders_excluded.select("order_id", "customer_id", "order_purchase_timestamp"),
    "order_id", "inner"
)

In [11]:
# Fonction pour calculer le chiffre d'affaires d'un segment donné
def calculer_ca(df, nom):
    resultat = df.agg(
        _round(_sum("price"), 2).alias("ca_produits"),
        _round(_sum("freight_value"), 2).alias("total_frais_livraison"),
        _round(_sum("price") + _sum("freight_value"), 2).alias("ca_total")
    )
    print(f"=== {nom} ===")
    resultat.show()
    return resultat

ca_delivered = calculer_ca(df_gold_revenue_delivered, "Revenu reconnu (delivered)")
ca_in_transit = calculer_ca(df_gold_revenue_in_transit, "Revenu en transit (payé, non livré)")
ca_excluded = calculer_ca(df_gold_revenue_excluded, "Exclu (annulé/indisponible)")


=== Revenu reconnu (delivered) ===
+-------------+---------------------+-------------+
|  ca_produits|total_frais_livraison|     ca_total|
+-------------+---------------------+-------------+
|1.322149811E7|           2198275.64|1.541977375E7|
+-------------+---------------------+-------------+

=== Revenu en transit (payé, non livré) ===
+-----------+---------------------+---------+
|ca_produits|total_frais_livraison| ca_total|
+-----------+---------------------+---------+
|  272902.63|             42850.65|315753.28|
+-----------+---------------------+---------+

=== Exclu (annulé/indisponible) ===
+-----------+---------------------+---------+
|ca_produits|total_frais_livraison| ca_total|
+-----------+---------------------+---------+
|   97242.96|             10783.25|108026.21|
+-----------+---------------------+---------+



In [12]:
# Sauvegarde en zone gold, avec séparation claire par segment
df_gold_revenue_delivered.write.mode("overwrite").parquet("../data/gold/account/revenue_delivered/")
df_gold_revenue_in_transit.write.mode("overwrite").parquet("../data/gold/account/revenue_in_transit/")
df_gold_revenue_excluded.write.mode("overwrite").parquet("../data/gold/account/revenue_excluded/")

In [20]:
from pyspark.sql.functions import col, count

# Combien de paiements par commande ?
payments_per_order = df_payments.groupBy("order_id").agg(count("*").alias("nb_paiements"))

payments_per_order.groupBy("nb_paiements").count().orderBy("nb_paiements").show()

+------------+-----+
|nb_paiements|count|
+------------+-----+
|           1|96479|
|           2| 2382|
|           3|  301|
|           4|  108|
|           5|   52|
|           6|   36|
|           7|   28|
|           8|   11|
|           9|    9|
|          10|    5|
|          11|    8|
|          12|    8|
|          13|    3|
|          14|    2|
|          15|    2|
|          19|    2|
|          21|    1|
|          22|    1|
|          26|    1|
|          29|    1|
+------------+-----+



In [ ]:
from pyspark.sql.functions import col

exemplo_order = df_payments.filter(col("order_id") ==
    df_payments.groupBy("order_id").count().orderBy(col("count").desc()).first()["order_id"]
)
exemplo_order.orderBy("payment_sequential").show(29)

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|fa65dad1b0e818e3c...|                 1|     voucher|                   1|         3.71|
|fa65dad1b0e818e3c...|                 2|     voucher|                   1|         8.51|
|fa65dad1b0e818e3c...|                 3|     voucher|                   1|         2.95|
|fa65dad1b0e818e3c...|                 4|     voucher|                   1|        29.16|
|fa65dad1b0e818e3c...|                 5|     voucher|                   1|         0.66|
|fa65dad1b0e818e3c...|                 6|     voucher|                   1|         5.02|
|fa65dad1b0e818e3c...|                 7|     voucher|                   1|         0.32|
|fa65dad1b0e818e3c...|                 8|     voucher|                   1|        26.02|
|fa65dad1b

In [23]:
from pyspark.sql.functions import sum as _sum, when, col

df_payment_breakdown = df_payments.groupBy("payment_type").agg(
    _sum("payment_value").alias("valeur_totale"),
    count("*").alias("nb_transactions")
).orderBy(col("valeur_totale").desc())

df_payment_breakdown.show()

+------------+--------------------+---------------+
|payment_type|       valeur_totale|nb_transactions|
+------------+--------------------+---------------+
| credit_card|1.2542084189999327E7|          76795|
|      boleto|  2869361.2699999753|          19784|
|     voucher|   379436.8700000007|           5775|
|  debit_card|   217989.7900000001|           1529|
| not_defined|                 0.0|              3|
+------------+--------------------+---------------+



In [28]:
from pyspark.sql.functions import sum as _sum, max as _max, collect_set, concat_ws

df_payments_agg = df_payments.groupBy("order_id").agg(
    _sum("payment_value").alias("valeur_totale_payee"),
    _max("payment_installments").alias("max_parcelles"),
    concat_ws(",", collect_set("payment_type")).alias("types_paiement")
)

df_payments_agg.orderBy(col("valeur_totale_payee").desc()).show(5)

+--------------------+-------------------+-------------+--------------+
|            order_id|valeur_totale_payee|max_parcelles|types_paiement|
+--------------------+-------------------+-------------+--------------+
|03caa2c082116e1d3...|           13664.08|            1|   credit_card|
|736e1922ae60d0d6a...|            7274.88|            1|        boleto|
|0812eb902a67711a1...|            6929.31|            8|   credit_card|
|fefacc66af859508b...|            6922.21|            1|        boleto|
|f5136e38d1a14a4db...|            6726.66|            1|        boleto|
+--------------------+-------------------+-------------+--------------+
only showing top 5 rows


In [29]:
df_orders_for_payment_analysis = df_orders.filter(
    ~col("order_status").isin(["canceled", "unavailable"])
)

df_gold_payments = df_payments_agg.join(
    df_orders_for_payment_analysis.select("order_id", "customer_id", "order_status", "order_purchase_timestamp"),
    "order_id",
    "inner"
)

In [32]:
from pyspark.sql.functions import avg, round as _round

df_panier_moyen = df_gold_payments.agg(
    _round(avg("valeur_totale_payee"), 2).alias("panier_moyen")
)
df_panier_moyen.show()

df_gold_payments.groupBy("types_paiement").agg(
    count("*").alias("nb_commandes"),
    _round(avg("valeur_totale_payee"), 2).alias("panier_moyen")
).orderBy(col("nb_commandes").desc()).show()

df_gold_payments.agg(
    _round(avg("max_parcelles"), 2).alias("parcelles_moyennes")
).show()

+------------+
|panier_moyen|
+------------+
|      160.27|
+------------+

+--------------------+------------+------------+
|      types_paiement|nb_commandes|panier_moyen|
+--------------------+------------+------------+
|         credit_card|       73407|      166.32|
|              boleto|       19539|      144.67|
| credit_card,voucher|        2210|       148.9|
|             voucher|        1535|      105.23|
|          debit_card|        1514|      140.27|
|credit_card,debit...|           1|      152.82|
+--------------------+------------+------------+

+------------------+
|parcelles_moyennes|
+------------------+
|              2.93|
+------------------+



In [33]:
# Répartition par type de paiement : valeur totale et nombre de transactions
df_gold_payment_breakdown = df_payments.groupBy("payment_type").agg(
    _round(_sum("payment_value"), 2).alias("valeur_totale"),
    count("*").alias("nb_transactions"),
    _round(avg("payment_value"), 2).alias("valeur_moyenne")
).orderBy(col("valeur_totale").desc())

df_gold_payment_breakdown.show()

+------------+-------------+---------------+--------------+
|payment_type|valeur_totale|nb_transactions|valeur_moyenne|
+------------+-------------+---------------+--------------+
| credit_card|1.254208419E7|          76795|        163.32|
|      boleto|   2869361.27|          19784|        145.03|
|     voucher|    379436.87|           5775|          65.7|
|  debit_card|    217989.79|           1529|        142.57|
| not_defined|          0.0|              3|           0.0|
+------------+-------------+---------------+--------------+



In [34]:
from pyspark.sql import Window
from pyspark.sql.functions import sum as _sum

total_geral = df_payments.agg(_sum("payment_value")).collect()[0][0]

df_gold_payment_breakdown = df_gold_payment_breakdown.withColumn(
    "pourcentage_valeur",
    _round((col("valeur_totale") / total_geral) * 100, 2)
)

df_gold_payment_breakdown.show()

+------------+-------------+---------------+--------------+------------------+
|payment_type|valeur_totale|nb_transactions|valeur_moyenne|pourcentage_valeur|
+------------+-------------+---------------+--------------+------------------+
| credit_card|1.254208419E7|          76795|        163.32|             78.34|
|      boleto|   2869361.27|          19784|        145.03|             17.92|
|     voucher|    379436.87|           5775|          65.7|              2.37|
|  debit_card|    217989.79|           1529|        142.57|              1.36|
| not_defined|          0.0|              3|           0.0|               0.0|
+------------+-------------+---------------+--------------+------------------+



In [35]:
df_payments.filter((col("payment_type") == "voucher") & (col("payment_value") == 0)).count()

6

In [37]:
df_gold_payment_breakdown.write.mode("overwrite").parquet("../data/gold/account/payment_breakdown/")

In [38]:
from pyspark.sql.functions import avg, round as _round, col

df_installments_by_type = df_payments.groupBy("payment_type").agg(
    _round(avg("payment_installments"), 2).alias("parcelles_moyennes"),
    count("*").alias("nb_transactions")
).orderBy(col("nb_transactions").desc())

df_installments_by_type.show()

+------------+------------------+---------------+
|payment_type|parcelles_moyennes|nb_transactions|
+------------+------------------+---------------+
| credit_card|              3.51|          76795|
|      boleto|               1.0|          19784|
|     voucher|               1.0|           5775|
|  debit_card|               1.0|           1529|
| not_defined|               1.0|              3|
+------------+------------------+---------------+



In [46]:
from pyspark.sql.functions import sum as _sum, avg, count, round as _round, col

# Calcul du total général pour le pourcentage
total_general_clients = df_payments_agg.join(
    df_orders.select("order_id", "customer_id"), "order_id", "inner"
).agg(_sum("valeur_totale_payee")).collect()[0][0]

df_payments_by_customer_state = df_payments_agg.join(
    df_orders.select("order_id", "customer_id"), "order_id", "inner"
).join(
    df_customers.select("customer_id", "customer_state"), "customer_id", "inner"
).groupBy("customer_state").agg(
    _round(_sum("valeur_totale_payee"), 2).alias("valeur_totale"),
    count("*").alias("nb_commandes"),
    _round(avg("valeur_totale_payee"), 2).alias("panier_moyen")
).withColumn(
    "pourcentage_valeur",
    _round((col("valeur_totale") / total_general_clients) * 100, 2)
).orderBy(col("valeur_totale").desc())

df_payments_by_customer_state.show(30)

+--------------+-------------+------------+------------+------------------+
|customer_state|valeur_totale|nb_commandes|panier_moyen|pourcentage_valeur|
+--------------+-------------+------------+------------+------------------+
|            SP|   5998226.96|       41745|      143.69|             37.47|
|            RJ|   2144379.69|       12852|      166.85|             13.39|
|            MG|   1872257.26|       11635|      160.92|              11.7|
|            RS|    890898.54|        5466|      162.99|              5.57|
|            PR|    811156.38|        5045|      160.78|              5.07|
|            SC|    623086.43|        3637|      171.32|              3.89|
|            BA|    616645.82|        3380|      182.44|              3.85|
|            DF|    355141.08|        2140|      165.95|              2.22|
|            GO|    350092.31|        2020|      173.31|              2.19|
|            ES|    325967.55|        2033|      160.34|              2.04|
|           

In [44]:
from pyspark.sql.functions import countDistinct

# Receita por vendedor (usando price de order_items, que já está no nível certo)
df_revenue_by_seller_state = df_items.join(
    df_sellers.select("seller_id", "seller_state"), "seller_id", "inner"
).groupBy("seller_state").agg(
    _round(_sum("price"), 2).alias("valeur_totale"),
    count("*").alias("nb_items"),
    _round(avg("price"), 2).alias("prix_moyen")
).orderBy(col("valeur_totale").desc())

df_revenue_by_seller_state.show(30)

+------------+-------------+--------+----------+
|seller_state|valeur_totale|nb_items|prix_moyen|
+------------+-------------+--------+----------+
|          SP|   8753396.21|   80342|    108.95|
|          PR|   1261887.21|    8671|    145.53|
|          MG|   1011564.74|    8827|     114.6|
|          RJ|    843984.22|    4818|    175.17|
|          SC|    632426.07|    4075|     155.2|
|          RS|    378559.54|    2199|    172.15|
|          BA|    285561.56|     643|    444.11|
|          DF|     97749.48|     899|    108.73|
|          PE|     91493.85|     448|    204.23|
|          GO|     66399.21|     520|    127.69|
|          ES|     47689.61|     372|     128.2|
|          MA|     36408.95|     405|      89.9|
|          CE|     20240.64|      94|    215.33|
|          PB|      17095.0|      38|    449.87|
|          MT|     17070.72|     145|    117.73|
|          RN|       9992.6|      56|    178.44|
|          MS|      8551.69|      50|    171.03|
|          RO|      

In [45]:
from pyspark.sql.functions import create_map, lit
from itertools import chain

region_map = {
    "AC": "Norte", "AP": "Norte", "AM": "Norte", "PA": "Norte", "RO": "Norte", "RR": "Norte", "TO": "Norte",
    "AL": "Nordeste", "BA": "Nordeste", "CE": "Nordeste", "MA": "Nordeste", "PB": "Nordeste",
    "PE": "Nordeste", "PI": "Nordeste", "RN": "Nordeste", "SE": "Nordeste",
    "DF": "Centro-Oeste", "GO": "Centro-Oeste", "MT": "Centro-Oeste", "MS": "Centro-Oeste",
    "ES": "Sudeste", "MG": "Sudeste", "RJ": "Sudeste", "SP": "Sudeste",
    "PR": "Sul", "RS": "Sul", "SC": "Sul"
}

mapping_expr = create_map([lit(x) for x in chain(*region_map.items())])

df_payments_by_customer_region = df_payments_by_customer_state.withColumn(
    "region", mapping_expr[col("customer_state")]
).groupBy("region").agg(
    _round(_sum("valeur_totale"), 2).alias("valeur_totale"),
    _sum("nb_commandes").alias("nb_commandes")
).orderBy(col("valeur_totale").desc())

df_payments_by_customer_region.show()

+------------+-------------+------------+
|      region|valeur_totale|nb_commandes|
+------------+-------------+------------+
|     Sudeste|1.034083146E7|       68265|
|         Sul|   2325141.35|       14148|
|    Nordeste|   1898479.44|        9394|
|Centro-Oeste|   1029797.52|        5782|
|       Norte|    414622.35|        1851|
+------------+-------------+------------+

